<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c40-header.png" alt="Nextia Learning · Experiments and Causal Impact: Did Our Change Work?" width="100%">

# Solution: Sample size and power

**[Sample size and power](https://learning.nextia-ai.com/courses/experiments/m02/sample-size-and-power/)** · Experiments and Causal Impact: Did Our Change Work? · Module 2, lesson 3 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** compute how many customers a test needs to find an effect of a chosen size, check that number by simulating many fake tests, and turn it into a number of days.

| | |
|---|---|
| **Time** | About 40 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | The primary metric, from [Choose the metrics](https://learning.nextia-ai.com/courses/experiments/m02/choose-the-metrics/); a p-value and a 95% interval, from the mathematics course: [Compare alternatives](https://learning.nextia-ai.com/courses/math/m05/compare-alternatives/). |
| **Tested** | Python 3.12 with pandas 3.1.1, SciPy 1.15.3, statsmodels 0.15.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/experiments/m02/sample-size-and-power/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C40/M02-L03-sample-size-and-power/sample-size-and-power-solution.ipynb).

## Setup: install the packages

Run the next cell first. It installs the versions that the course uses: pandas 3.1.1, SciPy 1.15.3 and statsmodels 0.15.0, and matplotlib 3.11.2 for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with these versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 "scipy==1.15.3; python_version < '3.14'" "scipy==1.18.1; python_version >= '3.14'" statsmodels==0.15.0 matplotlib==3.11.2
import importlib.metadata as _md
for _name in ["pandas", "scipy", "statsmodels", "matplotlib"]:
    print(_name, _md.version(_name))

## Setup: get the practice data

Run the next cell. It downloads the simulator's data for this lesson (less than 3 MB) into a folder `larkfield-experiments/data`, and checks each file's checksum. It needs an internet connection and no account.

You should see `Ready: 2 files in data/.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-experiments` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield data is synthetic: the course simulator made it. It has no real people. Each folder has `truth.json`: the true effects that the simulator knows. A real test never has this file.

In [ ]:
# Setup: download the course data into larkfield-experiments/data/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

DATA = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C40/data/"
FILES = {
    "delivery/customers.csv.gz": "c22521f272d36b36fd26234d53d04aac1c8be347a5014ee3e2aeeffcc34a0c19",
    "delivery/truth.json": "6f088f29a112de123bb1da3bac0940a28c5adb706e9f24007686c373c14ab6a4"
}
LOCAL = os.environ.get("NX_DATA_LOCAL")  # authors only: a local copy of the labs data
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("larkfield-experiments").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    if LOCAL:
        shutil.copy(Path(LOCAL) / name, path)
    else:
        download(DATA + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-experiments and run again.")
os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in data/.")

## Setup: helpers

The next cell loads pandas as `pd`, NumPy as `np`, SciPy's `stats` and statsmodels' formula interface as `smf`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `truth(name)` reads a scenario's `truth.json`: the simulator's true effects.

Run the cell. It prints nothing.

In [ ]:
# Helpers: checks that do not show the answer, and the simulator's truth.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
pd.set_option("display.precision", 4)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def truth(name):
    """The simulator's true effects for a scenario (a real test never has this)."""
    return json.loads(open(f"data/{name}/truth.json").read())

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. How many customers for the reminder email?

Rafael plans the reminder-email test. Today, 9% of the customers who get no email order in the next 14 days. He wants to find an effect of **1 percentage point** (9% to 10%), if it is there. That is the **minimum detectable effect**. He wants **80% power**: a test of this size finds a real effect of 1 point in 80 of 100 tests. And he uses the usual **alpha of 0.05**: "found" means p < 0.05.

statsmodels computes the size in two steps. `proportion_effectsize` turns 9% and 10% into one standard size of the effect. `NormalIndPower().solve_power` finds the customers per group.

> **Predict.** About how many customers per group: 1,000, 10,000 or 100,000?

In [ ]:
import math
from statsmodels.stats.power import NormalIndPower, TTestIndPower
from statsmodels.stats.proportion import proportion_effectsize

baseline, mde = 0.09, 0.01
effect_size = abs(proportion_effectsize(baseline + mde, baseline))
n = math.ceil(NormalIndPower().solve_power(effect_size=effect_size, power=0.8, alpha=0.05, ratio=1.0))
print(f"standard effect size {effect_size:.4f}: {n:,} customers per group, {2 * n:,} in total")

> **Check.** You should see `13,488 customers per group`.

## 2. Power at other sizes

Power is not "yes or no". A smaller test still finds the effect sometimes; a bigger one finds it more often. The next cell computes the power at half the size, the size and twice the size.

In [ ]:
for size in [n // 2, round(n * 0.75), n, round(n * 1.5), 2 * n]:
    power = NormalIndPower().power(effect_size=effect_size, nobs1=size, alpha=0.05, ratio=1.0)
    print(f"{size:>7,} per group: power {power:.2f}")

> **Check.** You should see power `0.51` at 6,744 per group, `0.80` at 13,488 and `0.98` at 26,976.

## 3. Check the formula: run 200 fake tests

A formula rests on assumptions. A simulation checks it. The next cell makes 200 fake tests of the planned size. In each one, A's customers order with a chance of 9% and B's with 10%: the true effect is exactly 1 point. Then it runs Welch's t-test, the same test as `xp analyse`, and counts the tests with p < 0.05.

> **Predict.** About how many of the 200 tests find the effect?

In [ ]:
def fake_test(seed, n, baseline, effect):
    rng = np.random.default_rng(seed)
    a = (rng.random(n) < baseline).astype(float)  # 1 = ordered, 0 = did not
    b = (rng.random(n) < baseline + effect).astype(float)
    return b.mean() - a.mean(), stats.ttest_ind(b, a, equal_var=False).pvalue

results = pd.DataFrame([fake_test(1000 + i, n, 0.09, 0.01) for i in range(200)], columns=["difference", "p"])
print(f"found (p < 0.05): {(results.p < 0.05).mean():.1%} of 200 fake tests")
print(f"the differences go from {results.difference.min():+.4f} to {results.difference.max():+.4f}")

> **Check.** You should see `found (p < 0.05): 79.5% of 200 fake tests` and differences from +0.0029 to +0.0207.

The true effect is the same in every fake test, but each test sees a different difference: chance moves it. The tests that find the effect are the ones where chance did not pull the difference below about 0.7 points. The chart shows it.

In [ ]:
import matplotlib.pyplot as plt

plt.hist(100 * results.difference, bins=30, color="#9aa7c7")
plt.axvline(1.0, color="black", label="true effect: 1 point")
plt.axvline(0.70, color="crimson", linestyle="--", label="smaller than this: p > 0.05")
plt.xlabel("difference B − A (percentage points), 200 fake tests")
plt.legend()
plt.show()

## 4. Power curves

The next cell draws power against the customers per group, for three effects: half a point, 1 point and 1.5 points.

> **Predict.** To find half a point instead of 1 point, do you need twice as many customers, or more?

In [ ]:
sizes = np.arange(500, 40001, 500)
for effect in [0.005, 0.01, 0.015]:
    es = abs(proportion_effectsize(0.09 + effect, 0.09))
    plt.plot(sizes, NormalIndPower().power(effect_size=es, nobs1=sizes, alpha=0.05, ratio=1.0), label=f"{effect * 100:g} point(s)")
plt.axhline(0.8, color="grey", linestyle=":")
plt.xlabel("customers per group")
plt.ylabel("power")
plt.legend()
plt.show()
for effect in [0.005, 0.015]:
    es = abs(proportion_effectsize(0.09 + effect, 0.09))
    print(f"{effect * 100:g} point(s): {math.ceil(NormalIndPower().solve_power(effect_size=es, power=0.8, alpha=0.05)):,} per group")

> **Check.** You should see `0.5 point(s): 52,700 per group` and `1.5 point(s): 6,131 per group`.

Half the effect needs about four times the customers: the size grows with 1 ÷ effect².

## 5. From customers to days

About 4,000 customers a day reach 30 days without an order, so they can enter the test. The test needs 26,976 customers in total.

In [ ]:
per_day = 4000
days = math.ceil(2 * n / per_day)
weeks = math.ceil(days / 7)
print(f"{days} days; run {weeks} full week(s): {7 * weeks} days")

> **Check.** You should see `7 days; run 1 full week(s): 7 days`.

Run full weeks: customers who shop on a Sunday are not the same as those who shop on a Tuesday.

## 6. The free-delivery test: a metric with a large spread

Revenue per customer is not a 0/1 metric. Most customers spend nothing, and a few spend hundreds of dollars. For a mean, the size depends on the **standard deviation**. The next cell reads the free-delivery test's group A and computes the smallest effect that its 55,430 customers per group could find with 80% power.

In [ ]:
delivery = pd.read_csv("data/delivery/customers.csv.gz")
a = delivery.loc[delivery.variant == "A", "revenue"]
sd, mean = a.std(), a.mean()
d = TTestIndPower().solve_power(nobs1=len(a), power=0.8, alpha=0.05, ratio=1.0)
print(f"group A: {len(a):,} customers, mean {mean:.2f} dollars, standard deviation {sd:.2f}")
print(f"smallest effect with 80% power: {d * sd:.2f} dollars ({d * sd / mean:.1%})")
print(f"true effect (from the simulator): {truth('delivery')['revenue']['effect']:+.2f} dollars")
for pct in [1, 2, 3, 5]:
    size = math.ceil(TTestIndPower().solve_power(effect_size=pct / 100 * mean / sd, power=0.8, alpha=0.05))
    print(f"{pct}%: {size:>9,} per group, {math.ceil(2 * size / 3956):>4} days at 3,956 customers a day")

> **Check.** You should see a standard deviation of `73.22`, `smallest effect with 80% power: 1.23 dollars (4.9%)`, a true effect of `+0.70 dollars`, and `3%:   149,024 per group,   76 days`.

The true effect, 0.70 dollars, is smaller than the smallest effect the test can find with 80% power. The test is **under-powered** for it: over 200 seeds of the simulator, it finds the effect in only 41.5% of runs. The lesson page shows what to do about it.

## 7. Your turn: a smaller email test

> **Your turn.** Rafael now says that only an effect of **1.5 points** (9% to 10.5%) would pay for the email. Compute the customers per group for 80% power at alpha 0.05, with `proportion_effectsize` and `NormalIndPower`, rounded up with `math.ceil`. Put it in `n_15`.

In [ ]:
es_15 = abs(proportion_effectsize(0.105, 0.09))
n_15 = math.ceil(NormalIndPower().solve_power(effect_size=es_15, power=0.8, alpha=0.05, ratio=1.0))
print(n_15)

In [ ]:
expect('n_15', n_15, 6131)

> **Your turn.** At 1,000 customers a day (a smaller list), how many **full weeks** does the 1-point test (`n` customers per group) need? Put the number of weeks in `weeks_1000`.

In [ ]:
weeks_1000 = math.ceil(math.ceil(2 * n / 1000) / 7)
print(weeks_1000)

In [ ]:
expect('weeks_1000', weeks_1000, 4)

## Next

You can size a test and check the size by simulation. The [lesson page](https://learning.nextia-ai.com/courses/experiments/m02/sample-size-and-power/) explains what to do when the test would take a year, and has the knowledge checks that count toward your certificate. The next lesson writes all these choices in a plan, before the test starts.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Write the plan first](https://learning.nextia-ai.com/courses/experiments/m02/write-the-plan-first/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/experiments/m02/sample-size-and-power/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.